# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 01: TensorRT, Operator Fusion & CUDA Graphs from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In production LLM serving, naive PyTorch execution is severely CPU-overhead bound:
- In a Transformer layer: `LayerNorm -> Linear -> GELU -> Linear` launches 4 distinct CUDA kernels sequentially.
- Launching each GPU kernel takes ~5 to 10 microseconds of CPU overhead.
- Intermediate activations must be written to global GPU VRAM (HBM) and immediately read back by the next kernel, saturating memory bandwidth.

**High-Performance Compilers (TensorRT, torch.compile):**
1. **Operator Fusion:** Combines `Linear + GELU + Add` into a single fused GPU kernel, performing computation in ultra-fast SRAM without intermediate VRAM roundtrips!
2. **CUDA Graphs:** Records the entire execution graph once and replays it with a single CPU launch, slashing launch overhead to zero!

In [ ]:
import time
import numpy as np

# Step 1: Unfused Pipeline (Simulating Separate CUDA Kernel Launches & VRAM I/O)
N, D = 10000, 512
np.random.seed(42)
X = np.random.randn(N, D)
W = np.random.randn(D, D) * 0.05
b = np.random.randn(D) * 0.05

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * np.power(x, 3))))

# Benchmark Unfused: 3 separate operations
start = time.perf_counter()
for _ in range(20):
    t1 = X @ W           # Kernel 1: MatMul (writes to VRAM)
    t2 = t1 + b          # Kernel 2: Bias Add (reads/writes VRAM)
    out_unfused = gelu(t2) # Kernel 3: Non-linearity (reads/writes VRAM)
time_unfused = (time.perf_counter() - start) / 20

# Step 2: Fused Operator (Simulating Fused Kernel inside CPU/GPU Cache)
def fused_linear_bias_gelu(X, W, b):
    # Computes MatMul and directly applies bias + GELU in registers/cache
    z = X @ W + b
    return 0.5 * z * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (z + 0.044715 * np.power(z, 3))))

start = time.perf_counter()
for _ in range(20):
    out_fused = fused_linear_bias_gelu(X, W, b)
time_fused = (time.perf_counter() - start) / 20

print("--- Operator Fusion Performance Benchmark ---")
print(f"Unfused Kernel Chain Time: {time_unfused * 1000:.2f} ms")
print(f"Fused Operator Time:       {time_fused * 1000:.2f} ms")
print(f"Speedup from Fusion:       {time_unfused / max(1e-6, time_fused):.2f}x faster!")
print("Exact Match:", np.allclose(out_unfused, out_fused))

### 3. Step-by-Step Code Walkthrough
1. **Unfused Roundtrips:** In unfused execution, separate function calls create memory thrashing.
2. **Fused Register Arithmetic:** The fused function keeps intermediate representations in processor cache.
3. **Exact Accuracy:** Generates identical outputs with significantly lower memory bandwidth consumption.

---

### 4. Industry Applications & Trade-offs
- **TensorRT / torch.compile (Inductor):** The industry standard for maximizing throughput on NVIDIA H100/A100 server clusters.
- **Trade-off:** Ahead-Of-Time (AOT) compilation introduces warmup latency on the very first forward pass.

---

### 5. Top Interview Questions & Answers
**Q1: What is a CUDA Graph and how does it reduce inference latency?**
> **Answer:** Standard execution launches kernels from CPU to GPU one at a time via PCIe driver queues. For small batch sizes, CPU launch overhead (10μs per kernel) dominates actual GPU execution time (2μs). A CUDA Graph records the whole kernel dependency tree once, allowing the CPU to launch the entire multi-kernel sequence with a single IO call.

**Q2: What is the difference between Compute-Bound and Memory-Bound operations?**
> **Answer:** Large matrix multiplications ($O(N^3)$ operations on $O(N^2)$ data) are compute-bound (limited by Tensor Core TFLOPs). Elementwise operations like LayerNorm and GELU ($O(N)$ operations on $O(N)$ data) are memory-bound (limited by memory bandwidth). Operator fusion accelerates memory-bound operations by eliminating global memory loads.